# Databricks RAGアプリチュートリアル - 2. RAGエージェントの構築（Databricks Apps 方式）

このノートブックでは、`1_...` で作成した AI Search Index を活用する **Tool-calling RAG エージェント**を構築し、**Databricks Apps** にデプロイします。これは 2026年時点の Databricks 公式推奨のデプロイ方式です。

**このノートブックは Databricks SDK だけで完結します**（App の作成・デプロイ・クエリまで）。`databricks` CLI（`databricks bundle`）のインストールや認証設定は不要で、ノートブックを上から実行するだけで進められます。

> 📎 従来の **Model Serving 方式**（`agents.deploy()`）や **DABs（CLI）方式**との違いは、末尾の **Appendix A** を参照してください。旧手順のノートブックは `old/2_RAGエージェントの構築.ipynb` にあります。

## このノートブックで学習する内容

1. **MLflow AgentServer / ResponsesAgent** のアーキテクチャ
2. **`@invoke()` / `@stream()`** によるエージェント実装（LangGraph 統合）
3. エージェントのソースを **Workspace に配置**し、**SDK（`w.apps`）で作成・デプロイ**
4. デプロイした App の**動作確認**（ブラウザのチャットUI／プログラマティックなクエリは Appendix B）

## 実行環境・前提

- **サーバーレスコンピュート**での実行を想定
- `1_PDFのパースとベクトルインデックスの作成.ipynb` で AI Search Index が作成済み

## 参考リンク

- [Databricks Apps でエージェントをオーサリング・デプロイ（公式・推奨）](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/author-agent)
- [Apps SDK（databricks-sdk-py）](https://databricks-sdk-py.readthedocs.io/en/latest/workspace/apps/apps.html)
- [app-templates リポジトリ](https://github.com/databricks/app-templates)


## 1. アーキテクチャ：MLflow AgentServer と ResponsesAgent

Databricks Apps 方式では、エージェント本体が **MLflow `AgentServer`（FastAPI ベースの非同期サーバー）** として動作します。Model Serving エンドポイントは作成されず、**アプリ自体がエージェントのサービング実体**になります。

エージェントのロジックは、次の2つのモジュールレベル関数として実装します（クラスではなく**関数**である点が旧方式との大きな違いです）。

- **`@stream()`** … ストリーミング応答を返す非同期ジェネレータ。ロジックの本体をここに集約し、`ResponsesAgentStreamEvent` を `yield` します。
- **`@invoke()`** … 非ストリーミング応答。通常は `@stream()` を呼び出して `response.output_item.done` イベントだけ収集し、`ResponsesAgentResponse` にまとめて返します。

入出力は **OpenAI Responses API 形式**（`input` 配列）です。LangGraph の messages 形式との相互変換には MLflow のヘルパー（`to_chat_completions_input` など）を使います。

### デプロイの流れ（SDK 完結）

```
1. エージェントのソース一式を Workspace パスに書き出す（w.workspace.upload）
2. App を作成（w.apps.create_and_wait）— アクセスするリソース権限もここで宣言
3. ソースをデプロイ（w.apps.deploy_and_wait）
4. App の URL を取得し、ブラウザのチャットUIで動作確認
```

### 配置するファイル

```
<workspace-src-dir>/
├── app.yaml                  # Databricks Apps 起動設定（command / env）
└── agent_server/
    ├── __init__.py
    ├── agent.py              # ★エージェント本体（@invoke / @stream）
    ├── utils.py              # LangGraph → Responses イベント変換ヘルパー
    └── start_server.py       # AgentServer 起動エントリポイント
```

> 💡 リソース権限（LLM エンドポイント・AI Search Index へのアクセス）は `app.yaml` ではなく、手順3の **App 作成時（`resources=[...]`）** に宣言します。


## 2. ライブラリの準備

エージェントの実装・ローカル検証、および App の作成・デプロイに使うライブラリをインストールします。**`databricks-sdk` に App 管理 API が含まれます**（CLI は不要）。

In [ ]:
# 実装・検証・デプロイ用ライブラリ（databricks-sdk が App 管理 API を提供）
%pip install -U -qqqq "databricks-sdk>=0.40.0" "mlflow>=3.10.0" "databricks-agents>=1.9.3" "databricks-langchain>=0.17.0" "langgraph>=1.1.0"
dbutils.library.restartPython()

## 3. パラメータ設定（widget）

環境依存の値を widget で指定します。エージェントのソースは **Workspace パス**（`/Workspace/Users/<自分>/...`）に配置します（Databricks Apps はソースを Workspace ファイルシステムから読み込むため）。

In [ ]:
# パラメータ設定（widget から取得）
dbutils.widgets.text("CATALOG_NAME", "skato", "カタログ名")
dbutils.widgets.text("SCHEMA_NAME", "rag_workshop", "スキーマ名")
dbutils.widgets.text("VECTOR_INDEX_NAME", "chunked_document_vs_index", "AI Search Index名")
dbutils.widgets.text("LLM_ENDPOINT_NAME", "databricks-claude-sonnet-4-5", "LLMエンドポイント名")
dbutils.widgets.text("APP_NAME", "rag-agent-app", "Databricks App 名（小文字英数字とハイフン, 2〜30文字）")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VECTOR_INDEX_NAME = dbutils.widgets.get("VECTOR_INDEX_NAME")
LLM_ENDPOINT_NAME = dbutils.widgets.get("LLM_ENDPOINT_NAME")
APP_NAME = dbutils.widgets.get("APP_NAME")

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# AI Search Index のフルネーム
VS_INDEX_FULLNAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{VECTOR_INDEX_NAME}"

# エージェントのソースを配置する Workspace ディレクトリ（実行ユーザーのホーム配下）
ME = w.current_user.me().user_name
APP_SOURCE_DIR = f"/Workspace/Users/{ME}/{APP_NAME}_src"

print(f"AI Search Index: {VS_INDEX_FULLNAME}")
print(f"LLMエンドポイント: {LLM_ENDPOINT_NAME}")
print(f"アプリ名: {APP_NAME}")
print(f"ソース配置先(Workspace): {APP_SOURCE_DIR}")

## 4. エージェント本体（`agent_server/agent.py`）

エージェントのロジックを `@invoke()` / `@stream()` 関数として定義します。ここが Model Serving 方式の `ChatAgent` クラスに代わる中心部です。

**ポイント:**
- `from mlflow.genai.agent_server import invoke, stream` でデコレータを import
- `@stream()` にロジック本体を書き、`@invoke()` はそれを呼んで結果を集約
- `create_agent`（LangChain）で LLM + retriever ツールを束ねた LangGraph エージェントを構築
- `request.input`（Responses 形式）→ `to_chat_completions_input(...)` で LangChain messages に変換
- `agent.astream(...)` の出力を `process_agent_astream_events(...)`（`utils.py`）で `ResponsesAgentStreamEvent` に変換

> 💡 このセルではソース文字列を変数に用意するだけです。実際の書き出し（Workspace への upload）は手順8でまとめて行います。retriever の index 名・LLM 名は widget 値を埋め込みます。

In [ ]:
AGENT_PY = """# Databricks Apps 版 Tool-calling RAG エージェント（ResponsesAgent / AgentServer）
# - @invoke() / @stream() でエージェントを定義（Model Serving 版の ChatAgent クラスに相当）
# - LangGraph（create_agent）で LLM + AI Search retriever ツールを束ねる
import logging
from typing import AsyncGenerator

import mlflow
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langchain.agents import create_agent
from mlflow.genai.agent_server import invoke, stream
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    to_chat_completions_input,
)

from agent_server.utils import process_agent_astream_events

logger = logging.getLogger(__name__)
mlflow.langchain.autolog()

# システムプロンプト
SYSTEM_PROMPT = (
    "あなたは生成AI開発に関する専門的なアシスタントです。"
    "登録された AI Search Index を活用し、検索された文書の内容に基づいて、"
    "日本語で丁寧かつ正確に回答してください。不確かな情報は推測しないでください。"
)


def init_agent():
    # AI Search Index を参照する retriever ツール
    retriever_tool = VectorSearchRetrieverTool(
        index_name="__VS_INDEX_FULLNAME__",
        tool_description=(
            "生成AI開発に関する技術文書やベストプラクティスを検索するツール。"
            "GenAI開発ワークフロー、MLOps、エージェント設計などの質問に使用する。"
        ),
    )
    llm = ChatDatabricks(endpoint="__LLM_ENDPOINT_NAME__")
    return create_agent(model=llm, tools=[retriever_tool], system_prompt=SYSTEM_PROMPT)


@stream()
async def stream_handler(request: ResponsesAgentRequest) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
    agent = init_agent()
    messages = {"messages": to_chat_completions_input([i.model_dump() for i in request.input])}
    async for event in process_agent_astream_events(
        agent.astream(input=messages, stream_mode=["updates", "messages"])
    ):
        yield event


@invoke()
async def invoke_handler(request: ResponsesAgentRequest) -> ResponsesAgentResponse:
    # @stream() の結果から完了アイテムだけを集めて返す
    outputs = [
        event.item
        async for event in stream_handler(request)
        if event.type == "response.output_item.done"
    ]
    return ResponsesAgentResponse(output=outputs)
"""

AGENT_PY = AGENT_PY.replace("__VS_INDEX_FULLNAME__", VS_INDEX_FULLNAME).replace("__LLM_ENDPOINT_NAME__", LLM_ENDPOINT_NAME)
print("agent.py を準備しました（書き出しは手順8）")

## 5. ストリーム変換ヘルパー（`agent_server/utils.py`）

LangGraph の `astream`（`updates` / `messages`）の出力を、Responses API の `ResponsesAgentStreamEvent` に変換するヘルパーです。公式テンプレート（app-templates）の実装に準拠しています。

In [ ]:
UTILS_PY = """# LangGraph の astream 出力を Responses イベントに変換するヘルパー
import json
import logging
from typing import Any, AsyncGenerator, AsyncIterator

from langchain.messages import AIMessageChunk, ToolMessage
from mlflow.types.responses import (
    ResponsesAgentStreamEvent,
    create_text_delta,
    output_to_responses_items_stream,
)


async def process_agent_astream_events(
    async_stream: AsyncIterator[Any],
) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
    async for event in async_stream:
        if event[0] == "updates":
            for node_data in event[1].values():
                if len(node_data.get("messages", [])) > 0:
                    for msg in node_data["messages"]:
                        # ツール結果が非文字列なら JSON 文字列化
                        if isinstance(msg, ToolMessage) and not isinstance(msg.content, str):
                            msg.content = json.dumps(msg.content)
                    for item in output_to_responses_items_stream(node_data["messages"]):
                        yield item
        elif event[0] == "messages":
            try:
                chunk = event[1][0]
                if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                    yield ResponsesAgentStreamEvent(
                        **create_text_delta(delta=content, item_id=chunk.id)
                    )
            except Exception as e:
                logging.exception(f"Error processing agent stream event: {e}")
"""
print("utils.py を準備しました（書き出しは手順8）")

## 6. サーバー起動エントリポイント（`agent_server/start_server.py`）

`AgentServer` を初期化し、`agent.py` の `@invoke`/`@stream` を登録して起動するエントリポイントです。通常このファイルは編集不要で、テンプレートそのままです。

In [ ]:
START_SERVER_PY = """# AgentServer 起動エントリポイント（通常は編集不要）
from pathlib import Path

from dotenv import load_dotenv
from mlflow.genai.agent_server import AgentServer, setup_mlflow_git_based_version_tracking

# .env があれば読み込む（ローカル実行時の認証用）
load_dotenv(dotenv_path=Path(__file__).parent.parent / ".env", override=True)

# agent モジュールを import して @invoke / @stream 関数をサーバーに登録
import agent_server.agent  # noqa: E402

agent_server = AgentServer("ResponsesAgent", enable_chat_proxy=True)

# 複数ワーカー対応のため app をモジュールレベル変数として公開
app = agent_server.app  # noqa: F841
setup_mlflow_git_based_version_tracking()


def main():
    agent_server.run(app_import_string="agent_server.start_server:app")
"""
print("start_server.py を準備しました（書き出しは手順8）")

## 7. 依存関係（`pyproject.toml`）と Apps 起動設定（`app.yaml`）

- **`pyproject.toml`**: アプリコンテナ側の依存関係。Databricks Apps は `uv` を使うため `pyproject.toml` が標準。`[project.scripts]` の `start-server` が起動コマンドの実体。
- **`app.yaml`**: コンテナ起動時のコマンドと環境変数。**リソース権限（`resources`）はここには書きません**（手順9の App 作成時に宣言）。

In [ ]:
PYPROJECT_TOML = """[project]
name = "rag-agent-server"
version = "0.1.0"
description = "RAG agent on Databricks Apps (ResponsesAgent)"
readme = "README.md"
requires-python = ">=3.11"
dependencies = [
    "fastapi>=0.129.0",
    "uvicorn>=0.41.0",
    "mlflow>=3.10.0",
    "databricks-agents>=1.9.3",
    "databricks-langchain>=0.17.0",
    "langgraph>=1.1.0",
    "python-dotenv>=1.2.1",
    "opentelemetry-exporter-otlp-proto-grpc>=1.25.0",
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project.scripts]
start-server = "agent_server.start_server:main"
"""

APP_YAML = """command: ["uv", "run", "start-server"]

env:
  - name: MLFLOW_TRACKING_URI
    value: "databricks"
  - name: MLFLOW_REGISTRY_URI
    value: "databricks-uc"
"""
print("pyproject.toml / app.yaml を準備しました（書き出しは手順8）")

## 8. ソース一式を Workspace に書き出す

用意した各ファイルを、手順3の `APP_SOURCE_DIR`（Workspace パス）に配置します。`.py` 以外（`.yaml` / `.toml`）も Workspace ファイルとして置くため、`w.workspace.upload(..., format=ImportFormat.AUTO)` を使います。

In [ ]:
from databricks.sdk.service.workspace import ImportFormat

# ディレクトリ作成
w.workspace.mkdirs(APP_SOURCE_DIR)
w.workspace.mkdirs(f"{APP_SOURCE_DIR}/agent_server")

def put(path: str, text: str):
    """Workspace パスにテキストファイルを配置（.yaml/.toml/.py すべて AUTO で workspace file 化）"""
    w.workspace.upload(path, text.encode("utf-8"), format=ImportFormat.AUTO, overwrite=True)

put(f"{APP_SOURCE_DIR}/agent_server/__init__.py", "")
put(f"{APP_SOURCE_DIR}/agent_server/agent.py", AGENT_PY)
put(f"{APP_SOURCE_DIR}/agent_server/utils.py", UTILS_PY)
put(f"{APP_SOURCE_DIR}/agent_server/start_server.py", START_SERVER_PY)
put(f"{APP_SOURCE_DIR}/pyproject.toml", PYPROJECT_TOML)
put(f"{APP_SOURCE_DIR}/app.yaml", APP_YAML)

print("Workspace に書き出しました:")
for e in w.workspace.list(f"{APP_SOURCE_DIR}/agent_server"):
    print("  ", e.path)
for e in w.workspace.list(APP_SOURCE_DIR):
    print("  ", e.path)

## 9. App を作成（`w.apps.create_and_wait`）

SDK で App を作成します。エージェントがアクセスするリソース（**LLM の Serving Endpoint** と **AI Search Index**）の権限は、ここで `resources=[...]` として宣言します。作成すると App 専用のサービスプリンシパルが自動生成され、宣言したリソースへのアクセス権が付与されます。

- Serving Endpoint → `AppResourceServingEndpoint`（`CAN_QUERY`）
- AI Search Index → `AppResourceUcSecurable`（`securable_type=TABLE`, `permission=SELECT`）

> ⏳ `create_and_wait` は App のコンピュートが起動するまで待機します（数分）。既に同名 App があるとエラーになるため、その場合は手順10（デプロイ）に進むか、別名にしてください。

In [ ]:
from databricks.sdk.service.apps import (
    App, AppResource,
    AppResourceServingEndpoint,
    AppResourceServingEndpointServingEndpointPermission,
    AppResourceUcSecurable,
    AppResourceUcSecurableUcSecurableType,
    AppResourceUcSecurableUcSecurablePermission,
)

app_spec = App(
    name=APP_NAME,
    description="RAG agent (ResponsesAgent) on Databricks Apps",
    resources=[
        AppResource(
            name="llm_endpoint",
            serving_endpoint=AppResourceServingEndpoint(
                name=LLM_ENDPOINT_NAME,
                permission=AppResourceServingEndpointServingEndpointPermission.CAN_QUERY,
            ),
        ),
        AppResource(
            name="vector_index",
            uc_securable=AppResourceUcSecurable(
                securable_full_name=VS_INDEX_FULLNAME,
                securable_type=AppResourceUcSecurableUcSecurableType.TABLE,
                permission=AppResourceUcSecurableUcSecurablePermission.SELECT,
            ),
        ),
    ],
)

try:
    created = w.apps.create_and_wait(app=app_spec)
    print(f"App を作成しました: {created.name}")
    print(f"サービスプリンシパル: {created.service_principal_client_id}")
except Exception as e:
    # 既に存在する場合などはメッセージを表示して続行（手順10で再デプロイ可能）
    print(f"App 作成をスキップまたは失敗: {e}")
    print("既に同名 App が存在する場合は、そのまま手順10のデプロイに進めます。")

## 10. ソースをデプロイ（`w.apps.deploy_and_wait`）

手順8で Workspace に置いたソースを App にデプロイします。`mode=SNAPSHOT` はデプロイ時点のソースを固定コピーします。

> ⏳ `deploy_and_wait` はデプロイが完了（`SUCCEEDED`）するまで待機します（数分）。コード修正後に再実行すれば、新しいソースで再デプロイされます。

In [ ]:
from databricks.sdk.service.apps import AppDeployment, AppDeploymentMode

deployment = w.apps.deploy_and_wait(
    app_name=APP_NAME,
    app_deployment=AppDeployment(
        source_code_path=APP_SOURCE_DIR,
        mode=AppDeploymentMode.SNAPSHOT,
    ),
)
print(f"デプロイ完了: deployment_id={deployment.deployment_id}")
print(f"ステータス: {deployment.status}")

## 11. デプロイした App を動かしてみる

デプロイが完了すると、Databricks Apps 上でエージェントが起動します。最も簡単な確認方法は、**App の URL をブラウザで開き、付属のチャット UI で対話する**ことです（`AgentServer(enable_chat_proxy=True)` によりチャット UI が有効です）。

> 💡 ノートブックやアプリから**プログラム的に**エージェントを呼び出す方法（OAuth 認証が必要）は、末尾の **Appendix B** を参照してください。

In [ ]:
# App の URL を取得（ブラウザで開いてチャットUIで試す）
app = w.apps.get(name=APP_NAME)
print(f"App 名: {app.name}")
print(f"状態:   {app.app_status.state if app.app_status else 'N/A'}")
print(f"URL:    {app.url}")
print("↑ この URL をブラウザで開くと、チャット UI でエージェントと対話できます。")

## 12. （任意）ノートブック内でのエージェント動作確認

デプロイ前後にかかわらず、エージェントのロジックをこのノートブック内で直接実行して確認できます。`AGENT_PY` の中身と同じ構成で LangGraph エージェントを組み、呼び出します。

In [ ]:
# ノートブック内で LangGraph エージェントを直接実行（任意の動作確認）
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langchain.agents import create_agent

_retriever = VectorSearchRetrieverTool(
    index_name=VS_INDEX_FULLNAME,
    tool_description="生成AI開発に関する技術文書を検索するツール。",
)
_agent = create_agent(
    model=ChatDatabricks(endpoint=LLM_ENDPOINT_NAME),
    tools=[_retriever],
    system_prompt="あなたは生成AI開発の専門アシスタントです。日本語で、検索結果に基づき回答してください。",
)
_result = _agent.invoke({"messages": [{"role": "user", "content": "生成AIの開発ワークフローについて教えてください"}]})
for m in _result["messages"]:
    if getattr(m, "content", ""):
        print(f"[{getattr(m,'type','?')}] {m.content[:300]}")

## 13. まとめと次のステップ

### このノートブックで学んだこと

- **Databricks Apps 方式**でのエージェント構築（`@invoke()` / `@stream()` + `AgentServer`）
- LangGraph エージェントの Responses API 形式への統合（`utils.py` の変換ヘルパー）
- ソースを **Workspace に配置**し、**SDK（`w.apps.create_and_wait` / `deploy_and_wait`）で作成・デプロイ**（CLI 不要）
- リソース権限を **App 作成時に宣言**（LLM=CAN_QUERY, AI Search Index=SELECT）
- デプロイした App の動作確認（ブラウザのチャット UI）

### 次のステップ

1. **評価**: 姉妹ノートブック（MLflow 評価詳細版）で、合成データ生成・カスタムスコアラー・LLM-as-a-Judge・プロンプトレジストリを学ぶ
2. **モニタリング**: MLflow 3 のリアルタイムトレーシングでエージェントの挙動を追跡
3. **UI 提供**: `3_Webアプリケーションのデプロイ.ipynb` の Streamlit チャットUIから、この App を利用する

### 参考リンク

- [Databricks Apps でエージェントをオーサリング・デプロイ](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/author-agent)
- [Apps SDK（databricks-sdk-py）](https://databricks-sdk-py.readthedocs.io/en/latest/workspace/apps/apps.html)
- [app-templates リポジトリ](https://github.com/databricks/app-templates)

---

## Appendix A. 他のデプロイ方式との比較

このノートブックは **Databricks SDK 完結**（`w.apps.create` / `deploy`）でデプロイしましたが、他に2つの方式があります。

| 方式 | 概要 | このワークショップでの扱い |
|---|---|---|
| **SDK（本ノートブック）** | `w.apps.create_and_wait` + `w.apps.deploy_and_wait`。CLI 不要でノート完結 | ✅ 採用 |
| **DABs（CLI）** | `databricks.yml` を書き、`databricks bundle deploy` → `run`。CI/CD 向き | 補足（下記） |
| **Model Serving（旧）** | `log_model` → `register_model` → `agents.deploy()`。エンドポイント作成方式 | `old/2_RAGエージェントの構築.ipynb` |

### DABs（CLI）方式の補足

ローカル端末や CI で `databricks` CLI が使える場合は、`databricks.yml`（bundle 定義）を用意して次のように deploy できます（詳細は割愛。[DABs ドキュメント](https://docs.databricks.com/aws/en/dev-tools/bundles/) 参照）。

```bash
databricks bundle validate
databricks bundle deploy
databricks bundle run <app-key>
```

`databricks.yml` では、本ノートブックで `App(resources=[...])` に書いたリソースを `resources.apps.<key>.resources` に、`app.yaml` の command/env を `config` に記述します。**ノートブックから CLI 認証なしで完結させたい場合は、本編の SDK 方式が適しています。**

### Model Serving（旧）方式

`agents.deploy()` による Model Serving エンドポイントへのデプロイは、動作はしますが新規ユースケースでは非推奨です。手順の全体は `old/2_RAGエージェントの構築.ipynb` を参照してください。

---

## Appendix B. プログラマティックにクエリする（OAuth 認証）

ノートブックやアプリからコードでエージェントを呼び出したい場合の手順です。**本編（手順11）のブラウザ UI での確認だけなら、このセクションは不要です。**

### なぜ OAuth が必要か

Databricks Apps 上のエージェントへのクエリは **OAuth 認証が必須**です。ノートブック既定の `WorkspaceClient()` は非 OAuth の一時トークンで認証されるため、そのまま `apps/<name>` を叩くと `ValueError: Querying Databricks Apps requires OAuth authentication.` になります。

ノートブック内で完結させるには、**OAuth M2M（サービスプリンシパルの `client_id` / `client_secret`）** でクライアントを構築します。

### 事前準備

1. **サービスプリンシパル（SP）を作成**: Settings → Identity and access → Service principals → Manage
2. **OAuth シークレットを発行**: 対象 SP → **Secrets** → **Generate secret**（`client_id` = SP の application ID）
3. **アプリに `CAN_USE` 権限を付与**: 対象 App の Permissions で、この SP に `CAN_USE`（無いと 403）

下のセルは、SP の資格情報を **widget から入力**して認証します（未入力なら分かりやすく停止します）。

In [ ]:
# Databricks OpenAI クライアントをインストール（Appendix B を実行する場合のみ）
%pip install -U -qqqq databricks-openai
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI

APP_NAME = dbutils.widgets.get("APP_NAME")

# --- OAuth M2M の資格情報を widget で入力 ---
dbutils.widgets.text("WORKSPACE_URL", "", "Workspace URL (https://...)")
dbutils.widgets.text("SP_CLIENT_ID", "", "SP client_id")
dbutils.widgets.text("SP_CLIENT_SECRET", "", "SP client_secret")

workspace_url = dbutils.widgets.get("WORKSPACE_URL")
client_id = dbutils.widgets.get("SP_CLIENT_ID")
client_secret = dbutils.widgets.get("SP_CLIENT_SECRET")

assert workspace_url and client_id and client_secret, (
    "上部の widget に Workspace URL / SP client_id / SP client_secret を入力してください。"
    "（SP には対象 App の CAN_USE 権限が必要です）"
)

# OAuth M2M で WorkspaceClient を構築（この w は OAuth 認証を持つ）
oauth_w = WorkspaceClient(host=workspace_url, client_id=client_id, client_secret=client_secret)
client = DatabricksOpenAI(workspace_client=oauth_w)

# apps/<app-name> 形式でクエリ（input は OpenAI Responses API 形式）
response = client.responses.create(
    model=f"apps/{APP_NAME}",
    input=[{"role": "user", "content": "生成AIの開発ワークフローについて教えてください"}],
)
print(response)

# --- 本番推奨: 資格情報を Databricks Secrets から取得（平文を避ける）---
# oauth_w = WorkspaceClient(
#     host=dbutils.secrets.get("<scope>", "workspace_url"),
#     client_id=dbutils.secrets.get("<scope>", "sp_client_id"),
#     client_secret=dbutils.secrets.get("<scope>", "sp_client_secret"),
# )